> 在线讲义：[第 30 章 · 从局部优化到完整请求](https://wikieden.github.io/ai-infra-from-constraints/lessons/30-request-view.html) · 本书《从约束推导 AI Infra》

# 第 30 章 · 从局部优化到完整请求

配套讲义：仓库里 `ai-infra/notes/lessons/30-request-view.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。纯 Python，不需要 GPU，也不下载模型。练习题不在这里算，先在讲义里自己做。

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 Amdahl | $1/((1-f) + f/s)$ | `amdahl(0.2, s)` | 1.11 / 1.22 / 1.25 |
| 第二步 关键路径 | $\max$ | `dag_time(10, a, b, 10)` | 80 / 60 / 110 μs |
| 第三步 请求 | 计数 | `36 * 32`、`13.6 / 851` | 1152 次、1.6% |

Amdahl、两分支的关键路径、Qwen3-8B 一次请求的 SwiGLU 替换取自 AII 第 5 章 §5.6。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

**函数 `amdahl`**：热点占 f、加速 s 倍时的整体加速比。

In [ ]:
def amdahl(f: float, s: float) -> float:
    """热点占 f、加速 s 倍时的整体加速比。"""
    return 1 / ((1 - f) + f / s)

**函数 `dag_time`**：准备后 A、B 并行，收尾等两者都完成。

In [ ]:
def dag_time(prep: float, a: float, b: float, tail: float) -> float:
    """准备后 A、B 并行，收尾等两者都完成。"""
    return prep + max(a, b) + tail

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步：热点占 20%")
for s in (2, 10, float("inf")):
    print(f"  加速 {s} 倍：整体 {amdahl(0.2, s):.2f} 倍（新时间 {1 / amdahl(0.2, s):.2f}）")

In [ ]:
print("\n第二步：两条分支")
print(f"  原 {dag_time(10, 60, 40, 10):.0f} μs；A 快 4 倍 {dag_time(10, 15, 40, 10):.0f} μs；A 再快也停在 {dag_time(10, 0, 40, 10):.0f} μs")
print(f"  A 抢带宽让 B 变 90 μs：{dag_time(10, 15, 90, 10):.0f} μs")

In [ ]:
print("\n第三步：Qwen3-8B，7239 入 32 出，36 层")
print(f"  SwiGLU 调用 {36 * 32}：prefill {36} 次共 {36 * 7239:,} 行，decode {36 * 31} 次各 1 行")
print(f"  原激活合计 13.6 ms，占 851 ms 的 {13.6 / 851:.1%}；decode 2.4 → 0.92 ms（{2.4 / 0.92:.1f} 倍）")
print(f"  预期省 ≈ 1.4 ms，实测配对中位数 1.3 ms（{1.3 / 817:.2%}）")

## 3. 代码里才看得到的

1. `amdahl(0.2, float("inf"))` 直接得 1.25：Python 里 `f / inf` 是 0。
2. `dag_time(10, 40, 40, 10)` 和 `dag_time(10, 1, 40, 10)` 都是 60：A 低于 40 以后再缩毫无作用。
3. 原文"decode 合计节省约 1.45 ms"用的是未取整的阶段时间；拿表里取整的 2.4 和 0.92 相减是 1.48。

## 4. 自己改一改

- 把 `dag_time` 推广成任意多条分支，再加一条"A 每快 1 μs，B 慢 0.5 μs"的共享带宽规则，求最优的 A。
- 用 `amdahl` 画 $f$ = 0.016（Qwen3 的激活占比）时 $s$ 从 1 到 100 的曲线。
- 把第三步改成生成 1024 个 token，激活节省占请求的比例会变成多少（假设每步其他工作不变）？

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 地铁再快，走路那段还得走；
> 炖肉快了看煮饭，抢了燃气饭更晚；
> 雨刷快两倍半，整趟只快千分之二；
> 先数次数，再算每次，最后看谁在关键路径上。